**Identifying an unknown gas using the ideal gas law.**
A cylinder contains an unknown pure gas at a fixed pressure of 2.0 atm
and a known sample mass of 50 g.
Temperature vs. volume measurements are recorded in `gas.csv`.
The **ideal gas law** $PV = nRT$ predicts that volume is linear in temperature
at constant pressure and moles:

$$V = \frac{nR}{P}\,T$$

The slope of the $V$ vs. $T$ line is $nR/P$, from which the number of moles
$n = P\,\cdot\,\mathrm{slope}\,/\,R$ can be extracted.
Dividing the sample mass by $n$ gives the molar mass in g/mol,
which identifies the element.

In [ ]:
"""identify_element.ipynb"""

# Cell 01 - Load the temperature/volume data from CSV
# Column 0: temperature (Celsius), Column 1: volume (Liters)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

file_name = "gas.csv"
data = np.genfromtxt(file_name, delimiter=",")
pd.DataFrame(data, columns=["Temperature (C)", "Volume (L)"])

**Converting to SI units.**
The ideal gas law uses SI units throughout:
temperature in Kelvin ($T_K = T_C + 273.15$),
volume in cubic metres ($1\,\text{L} = 10^{-3}\,\text{m}^3$),
pressure in Pascals, and the gas constant
$R = 8.31446\,\text{J}\,\text{mol}^{-1}\,\text{K}^{-1}$.

In [ ]:
# Cell 02 - Convert to SI units: Celsius -> Kelvin, Litres -> cubic metres

temperature = data[:, 0] + 273.15  # Celsius to Kelvin
volume = data[:, 1] / 1000  # Litres to cubic metres
pd.DataFrame({"Temperature (K)": temperature, "Volume (m^3)": volume})

**Linear regression via the Gauss normal equations.**
The ordinary least-squares slope and intercept for $n$ data points $(x_i, y_i)$ are:

$$m = \frac{n\sum x_i y_i - \sum x_i \sum y_i}
          {n\sum x_i^2 - \left(\sum x_i\right)^2}
\qquad
b = \frac{\sum y_i - m\sum x_i}{n}$$

These minimize the sum of squared residuals $\sum(y_i - mx_i - b)^2$.

In [ ]:
# Cell 03 - Fit a line to the temperature vs. volume data


def fit_linear(x, y):
    """Return (slope, intercept) of the least-squares line through (x, y)."""
    n = len(x)
    m = n * np.sum(x * y) - np.sum(x) * np.sum(y)
    m /= n * np.sum(x**2) - np.sum(x) ** 2
    b = (np.sum(y) - m * np.sum(x)) / n
    return m, b


slope, yint = fit_linear(temperature, volume)
print(f"Slope (dV/dT) : {slope:.8f} m^3/K")
print(f"Y-intercept   : {yint:.8f} m^3")

**Extracting the number of moles.**
From $V = (nR/P)\,T$, the slope equals $nR/P$, so:

$$n = \frac{P \cdot \mathrm{slope}}{R}$$

The pressure is 2.0 atm converted to Pascals ($1\,\text{atm} = 101\,325\,\text{Pa}$).

In [ ]:
# Cell 04 - Solve for number of moles using PV=nRT -> n = P*slope/R

p = 2.0 * 101_325  # 2.0 atm -> Pascals
r = 8.31446261815324  # gas constant (J / mol / K)
n = p * slope / r  # moles of gas
print(f"Number of moles: {n:.8f} mol")

**Identifying the element.**
Molar mass $M = m_{\text{sample}} / n$ in g/mol.
Matching this to the periodic table identifies the element.

In [ ]:
# Cell 05 - Compute molar mass: M = sample_mass / moles

m_sample = 50  # grams (given)
atomic_mass = m_sample / n  # g/mol = u
print(f"Molar mass of unknown gas: {atomic_mass:.4f} u")

**Building the plotting domain.**
`np.linspace(0, 500)` creates 50 evenly spaced temperature values
from 0 to 500 K used to draw the best-fit line across the full range.

In [ ]:
# Cell 06 - Temperature domain for the best-fit line

t = np.linspace(0, 500)
print(t)

**Plotting temperature vs. volume.**
The red scatter points are the raw CSV measurements.
The blue line is the linear regression fit.
The title shows the computed molar mass so the element can be looked up
directly from the plot.

In [ ]:
# Cell 07 - Plot the data and best-fit line

plt.figure("identify_element", figsize=(7, 5))
plt.scatter(temperature, volume, color="red", zorder=5, label="Measurements")
plt.plot(t, slope * t + yint, label="Best-fit line")
plt.title(f"Unknown Gas ({atomic_mass:.3f} u)")
plt.xlabel("Temperature (K)")
plt.ylabel("Volume ($m^3$)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()